In [8]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters langchain-openai faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 6.3 MB/s eta 0:00:00a 0:00:01


In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader

In [9]:
loader = PyPDFLoader("https://cs229.stanford.edu/main_notes.pdf")
docs = loader.load()

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
documents= text_splitter.split_documents(docs)

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [14]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [ ]:
llm = ChatOpenAI()

In [24]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
     return_source_documents=True
)

In [25]:
query = "What is supervised learning?"
result = qa_chain({"query": query})
print(result["result"])

Supervised learning is a type of learning problem where an algorithm learns from a "training set" that includes both inputs (x) and their corresponding target outputs (y). The goal is for the learning algorithm to create a hypothesis (h) that can then predict the output (y) for new, unseen inputs (x).

It can be categorized into:
*   **Regression problems:** When the target variable (y) is continuous (e.g., predicting house prices).
*   **Classification problems:** When the target variable (y) can take on only a small number of discrete values (e.g., predicting if an animal is a dog or an elephant, or if a dwelling is a house or an apartment).


In [26]:
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:100]}...")

- Part I
Supervised learning
5...
- decision boundary it falls, and makes its prediction accordingly.
Here’s a diﬀerent approach. First,...
- 7
function h is called a hypothesis. Seen pictorially, the process is therefore
like this:
Training ...


#### Advanced RAG with Langchain